# Multimodal Retrieval-Augmented Generation (Multimodal RAG)

Welcome to this advanced tutorial on **Multimodal RAG**. In this notebook, we will learn how to build a RAG pipeline that can handle documents containing both **text** and **images** (such as charts, diagrams, and figures) using a unified vector space.

### What is Multimodal RAG? (Multimodal RAG kya hai?)
Standard RAG systems only process text. However, real-world documents (like financial statements, engineering manuals, textbooks) contain crucial information inside images, charts, and diagrams. **Multimodal RAG** allows us to retrieve relevant text paragraphs AND relevant images from our database, and pass both to a Multimodal/Vision LLM to generate a complete, context-aware answer.

### Why do we need a Unified Embedding Space (CLIP)?
To search for both text and images together, we use **CLIP (Contrastive Language-Image Pre-training)**. 
CLIP maps both text queries and visual images into the **same mathematical vector space**. This means we can convert a user's text question (e.g., *"Show me the revenue chart"*) into a vector, and find the closest matching image vectors (e.g., the actual JPG/PNG chart of revenue) directly using cosine similarity!

### Technology Stack & Environment:
- **CLIP Model**: `openai/clip-vit-base-patch32` for generating embeddings.
- **PDF Parser**: `PyMuPDF (fitz)` to extract text and image bytes.
- **Vector Database**: `FAISS` to store and query text and image vectors.
- **Vision LLM**: Groq's **`meta-llama/llama-4-scout-17b-16e-instruct`** (using your `GROQ_API_KEY`) for reasoning over the combined text and image context.

In [1]:
import os
import base64
import io
import fitz  # PyMuPDF for parsing PDF files
import torch
import numpy as np
from PIL import Image
from dotenv import load_dotenv
from sklearn.metrics.pairwise import cosine_similarity

# CLIP Model libraries
from transformers import CLIPProcessor, CLIPModel

# LangChain structures
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

# Load environment variables (.env)
load_dotenv()

print("All multimodal libraries imported successfully!")


e:\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Abhishek\AppData\Local\Temp\ipykernel_2836\1251236210.py:17: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


All multimodal libraries imported successfully!


## Phase 1: Initializing CLIP Model

We use Hugging Face's `transformers` library to load `openai/clip-vit-base-patch32`.

### What happens here? (Yahan kya ho rha hai?)
- **`CLIPModel`**: The actual neural network containing a Text Encoder and an Image Encoder.
- **`CLIPProcessor`**: Handles resizing, normalizing images, and tokenizing text inputs so the model can process them.
- **Model Download Note**: Running this for the first time will download ~600MB of model weights. We set the model to `eval()` mode because we are only using it for inference (generating embeddings), not training.

In [2]:
print("Loading CLIP model and processor (this may take a moment on first download)...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Put model in evaluation mode
clip_model.eval()
print(f"CLIP Model loaded successfully! Shared embedding space dimension: {clip_model.projection_dim}")


Loading CLIP model and processor (this may take a moment on first download)...


Loading weights: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 398/398 [00:00<00:00, 8187.70it/s]


CLIP Model loaded successfully! Shared embedding space dimension: 512


## Phase 2: Embedding Generation Functions

We define two functions: one for embedding images and one for embedding text.

### Why do we normalize embeddings? (Norm validation kyu zaroori hai?)
We divide the raw feature vectors by their magnitude (`features / features.norm(...)`). This normalizes each embedding vector to a length of `1`.
**Why?** When vectors are normalized, calculating **Cosine Similarity** is mathematically identical to a simple **Dot Product**. This speeds up database lookup times and ensures distance metrics behave consistently.

In [3]:
def embed_image(image_data) -> np.ndarray:
    """
    Generates a normalized CLIP embedding vector for a given image.
    Args:
        image_data: Path to image file (str) or a PIL Image object.
    """
    if isinstance(image_data, str):
        image = Image.open(image_data).convert("RGB")
    else:
        image = image_data.convert("RGB")
        
    # Preprocess image and convert to PyTorch tensors
    inputs = clip_processor(images=image, return_tensors="pt")
    
    with torch.no_grad():
        # Extract image feature vectors
        features = clip_model.get_image_features(**inputs)
        # DEFENSIVE CHECK: Handle cases where model output is wrapped in a BaseModelOutput class
        if not isinstance(features, torch.Tensor):
            if hasattr(features, "pooler_output") and features.pooler_output is not None:
                features = features.pooler_output
            elif hasattr(features, "last_hidden_state") and features.last_hidden_state is not None:
                features = features.last_hidden_state
            else:
                features = features[0]
        # Normalize to unit vector
        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().numpy()

def embed_text(text: str) -> np.ndarray:
    """
    Generates a normalized CLIP embedding vector for a text string.
    """
    # Tokenize and preprocess text
    inputs = clip_processor(
        text=text, 
        return_tensors="pt", 
        padding=True, 
        truncation=True,
        max_length=77  # CLIP's maximum token limit
    )
    
    with torch.no_grad():
        # Extract text feature vectors
        features = clip_model.get_text_features(**inputs)
        # DEFENSIVE CHECK: Handle cases where model output is wrapped in a BaseModelOutput class
        if not isinstance(features, torch.Tensor):
            if hasattr(features, "pooler_output") and features.pooler_output is not None:
                features = features.pooler_output
            elif hasattr(features, "last_hidden_state") and features.last_hidden_state is not None:
                features = features.last_hidden_state
            else:
                features = features[0]
        # Normalize to unit vector
        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().numpy()

print("Multimodal embedding functions loaded!")


Multimodal embedding functions loaded!


## Phase 3: Processing PDF (Extracting Text and Images)

We will load our PDF file page-by-page. For each page:
1. **Text**: Extract raw text, split it into chunks using `RecursiveCharacterTextSplitter`, embed each chunk with CLIP, and save it.
2. **Images**: Extract all embedded images using PyMuPDF. For each image:
   - Convert bytes to PIL Image.
   - Generate a unique ID (e.g., `page_1_img_0`).
   - **Encode to base64**: We store the image as a base64-encoded string in a Python dictionary. This is required because Vision LLMs receive images as base64 data URLs.
   - Generate CLIP image embedding and add it to our unified dataset.

### What happens if we don't save the actual base64 image data?
If we only save the embedding, we can find the image during retrieval, but we won't be able to show it to the LLM. The Vision LLM needs the actual image pixel data (via base64) to reason over its content (e.g. read the chart values).

In [4]:
def process_multimodal_pdf(pdf_file_path):
    """
    Loads PDF, extracts text chunks and images, encodes images to base64,
    and generates unified CLIP embeddings for all elements.
    """
    if not os.path.exists(pdf_file_path):
        # If sample PDF is missing, we create a placeholder warning
        raise FileNotFoundError(f"PDF file not found at: {pdf_file_path}. Please place your target PDF in this folder.")
        
    doc = fitz.open(pdf_file_path)
    all_docs = []
    all_embeddings = []
    image_data_store = {}  # Global lookup for image ID -> base64 string
    
    # Text splitter setup
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
    
    print(f"Processing multimodal PDF: {pdf_file_path} ({len(doc)} pages)")
    
    for i, page in enumerate(doc):
        # 1. PROCESS TEXT ON PAGE
        text = page.get_text()
        if text.strip():
            temp_doc = Document(page_content=text, metadata={"page": i, "type": "text"})
            text_chunks = splitter.split_documents([temp_doc])
            
            for chunk in text_chunks:
                embedding = embed_text(chunk.page_content)
                all_embeddings.append(embedding)
                all_docs.append(chunk)
                
        # 2. PROCESS IMAGES ON PAGE
        images_list = page.get_images(full=True)
        for img_index, img in enumerate(images_list):
            try:
                xref = img[0]
                base_image = doc.extract_image(xref)
                image_bytes = base_image["image"]
                
                # Convert bytes to PIL
                pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
                
                # Generate unique ID
                image_id = f"page_{i}_img_{img_index}"
                
                # Convert PIL Image to base64 string
                buffered = io.BytesIO()
                pil_image.save(buffered, format="PNG")
                img_base64 = base64.b64encode(buffered.getvalue()).decode()
                image_data_store[image_id] = img_base64
                
                # Embed image using CLIP
                embedding = embed_image(pil_image)
                all_embeddings.append(embedding)
                
                # Create image metadata document representation
                image_doc = Document(
                    page_content=f"[Image Location: {image_id}]",
                    metadata={
                        "page": i, 
                        "type": "image", 
                        "image_id": image_id,
                        "caption": f"Image extract {img_index} on page {i}"
                    }
                )
                all_docs.append(image_doc)
                
            except Exception as e:
                print(f"  âœ— Error processing image {img_index} on Page {i}: {e}")
                
    doc.close()
    print(f"Processing finished. Loaded {len(all_docs)} elements (text chunks + images).")
    return all_docs, np.array(all_embeddings), image_data_store

# Run extraction pipeline. 
# Replace this file path with your own PDF path if needed.
pdf_path = "multimodal_sample.pdf"

try:
    all_docs, embeddings_array, image_data_store = process_multimodal_pdf(pdf_path)
except Exception as e:
    print(f"Execution skipped: {e}")
    print("To execute, ensure a sample PDF file named 'multimodal_sample.pdf' is present in this directory.")
    # Create empty mock structures for running downstream cells without errors
    all_docs, embeddings_array, image_data_store = [], np.empty((0, 512)), {}


Processing multimodal PDF: multimodal_sample.pdf (19 pages)
Processing finished. Loaded 186 elements (text chunks + images).


## Phase 4: Unified Vector Store (FAISS)

We use **FAISS (Facebook AI Similarity Search)** to index our vectors. 
Since we already precomputed the CLIP vectors ourselves, we initialize FAISS using `FAISS.from_embeddings`. 
We pass `embedding=None` because we don't want FAISS to call a default embedding functionâ€”our custom vectors are already prepared.

In [5]:
if len(all_docs) > 0:
    print("Initializing FAISS vector store with precomputed CLIP embeddings...")
    # Wrap text and embeddings together as expected by LangChain API
    text_embeddings = [(doc.page_content, emb) for doc, emb in zip(all_docs, embeddings_array)]
    
    vector_store = FAISS.from_embeddings(
        text_embeddings=text_embeddings,
        embedding=None,  # We pass precomputed embeddings directly
        metadatas=[doc.metadata for doc in all_docs]
    )
    print("FAISS vector store initialized successfully!")
else:
    print("FAISS initialization skipped because no document elements were loaded.")
    vector_store = None


Initializing FAISS vector store with precomputed CLIP embeddings...


`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


FAISS vector store initialized successfully!


## Phase 5: Unified Multimodal Retrieval

To retrieve, we convert the user's text question into a CLIP vector using `embed_text`. We query the FAISS index to find the top `k` most similar items.

Because text and images are embedded in the *same* space, the returned items can contain both text documents and image document placeholders (e.g. `[Image Location: page_0_img_1]`).

In [6]:
def retrieve_multimodal(query: str, k: int = 3):
    """
    Encodes text query and performs similarity search across the FAISS index.
    """
    if not vector_store:
        print("Vector store is empty!")
        return []
        
    # Convert query to CLIP vector representation
    query_embedding = embed_text(query)
    
    # Query FAISS index using precomputed vector
    results = vector_store.similarity_search_by_vector(
        embedding=query_embedding,
        k=k
    )
    return results

print("Multimodal retriever function loaded!")


Multimodal retriever function loaded!


## Phase 6: Vision LLM Generation (Groq Llama 3.2 Vision)

Once we retrieve the relevant context (text + images), we need to feed both to our Vision LLM.

### How do we structure a Multimodal message for LangChain?
Instead of passing a simple string, we construct a `HumanMessage` where the content is a **list of dictionaries**:
1. **Text components**: `{"type": "text", "text": "Context information..."}`
2. **Image components**: `{"type": "image_url", "image_url": {"url": "data:image/png;base64,..."}}`

### Initializing Groq Client:
- We use **`meta-llama/llama-4-scout-17b-16e-instruct`** hosted on Groq.
- It requires your `GROQ_API_KEY` (loaded from `.env`).
- Temperature is set low (`0.1`) to prevent hallucination of factual numbers from charts.

In [10]:
# Initialize ChatGroq with Vision model
try:
    llm = ChatGroq(
    model_name="meta-llama/llama-4-scout-17b-16e-instruct",
    temperature=0.1
)

    print("Groq Vision LLM initialized successfully!")
except Exception as e:
    print(f"Error initializing ChatGroq: {e}")
    llm = None

def create_multimodal_message(query: str, retrieved_docs) -> HumanMessage:
    """
    Compiles retrieved text excerpts and base64-encoded image strings into a
    structured HumanMessage payload for the Vision LLM.
    """
    content = []
    
    # 1. Add user query instruction
    content.append({
        "type": "text",
        "text": f"You are an expert assistant. Answer the Question using the provided text snippets and images.\n\nQuestion: {query}\n"
    })
    
    # Separate text chunks and image chunks
    text_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "text"]
    image_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "image"]
    
    # 2. Add text context
    if text_docs:
        text_excerpts = []
        for doc in text_docs:
            page_num = doc.metadata.get("page", 0) + 1
            text_excerpts.append(f"[From PDF Page {page_num}]: {doc.page_content}")
            
        content.append({
            "type": "text",
            "text": "\nRelevant Text Excerpts:\n" + "\n\n".join(text_excerpts) + "\n"
        })
        
    # 3. Add retrieved images (base64 payloads)
    for doc in image_docs:
        image_id = doc.metadata.get("image_id")
        page_num = doc.metadata.get("page", 0) + 1
        
        if image_id and image_id in image_data_store:
            # Add label for image
            content.append({
                "type": "text",
                "text": f"\n[Image excerpt from PDF Page {page_num}]:\n"
            })
            # Add base64 image data url
            content.append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{image_data_store[image_id]}"
                }
            })
            
    # 4. Add final response guidance
    content.append({
        "type": "text",
        "text": "\nAnswer: Provide a concise and factual response. Cite the page numbers when referencing facts."
    })
    
    return HumanMessage(content=content)

print("Multimodal message creation wrapper loaded!")


Groq Vision LLM initialized successfully!
Multimodal message creation wrapper loaded!


## Phase 7: Orchestrating the Multimodal RAG Pipeline

We group the entire workflow into a single function: 
**Retrieve Context -> Build Message -> Invoke Vision LLM -> Output Answer**.

In [11]:
def run_multimodal_rag(query: str, k: int = 3) -> str:
    """
    Orchestrates unified retrieval and generation for multimodal inputs.
    """
    if not llm:
        return "Error: Groq LLM client is not initialized. Please verify your environment variables."
        
    # 1. Retrieve top-k documents (text or images)
    retrieved_docs = retrieve_multimodal(query, k=k)
    
    if not retrieved_docs:
        return "No context could be retrieved from the database to answer this question."
        
    # Print retrieval trace to monitor source attribution
    print(f"\n--- Retriever Trace for: '{query}' ---")
    for i, doc in enumerate(retrieved_docs):
        doc_type = doc.metadata.get("type", "unknown").upper()
        page_num = doc.metadata.get("page", 0) + 1
        print(f"  [{i+1}] MODALITY: {doc_type} | Page {page_num}")
        if doc_type == "TEXT":
            print(f"      Snippet: {doc.page_content[:120]}...")
            
    # 2. Build the multimodal message
    message = create_multimodal_message(query, retrieved_docs)
    
    # 3. Call LLM for generation
    print("Generating answer from Vision LLM...")
    try:
        response = llm.invoke([message])
        return response.content
    except Exception as e:
        return f"Error during generation: {e}"

print("Multimodal RAG Pipeline compiled!")


Multimodal RAG Pipeline compiled!


In [12]:
# Example queries to test the pipeline
if __name__ == "__main__":
    if vector_store:
        test_queries = [
            "What trends are shown in the revenue chart?",
            "Summarize the key information found in this PDF"
        ]
        
        for q in test_queries:
            print("\n========================================")
            ans = run_multimodal_rag(q, k=3)
            print(f"\nAnswer:\n{ans}")
            print("========================================\n")
    else:
        print("Execution skipped: To run, make sure 'multimodal_sample.pdf' is present and process the document.")




--- Retriever Trace for: 'What trends are shown in the revenue chart?' ---
  [1] MODALITY: TEXT | Page 8
      Snippet: 54.8
46.5
51.9
17.9
22.6
56.2
49.4
74.5
90.6
RAG-Sequence
44.0
55.8
44.9
53.4
15.3
21.5
57.2
47.5
between these dates an...
  [2] MODALITY: TEXT | Page 5
      Snippet: a correct answer being generated, which is not possible with standard extractive approaches, leading
5...
  [3] MODALITY: TEXT | Page 6
      Snippet: 15.1
19.7
38.2
41.6
64.0
81.1
RAG-Tok. 17.3
22.2
40.1
41.5
72.5
89.5
RAG-Seq. 14.7
21.4
40.8
44.2
to more effective marg...
Generating answer from Vision LLM...

Answer:
There is no revenue chart provided in the text excerpts. However, there are several numerical values listed on pages 6 and 8.

On page 6, the values for RAG-Tok. and RAG-Seq. appear to be increasing: 
- RAG-Tok.: 17.3, 22.2, 40.1, 41.5, 72.5, 89.5 
- RAG-Seq.: 14.7, 21.4, 40.8, 44.2 

On page 8, there is another list of numerical values: 
54.8, 46.5, 51.9, 17.9, 22.6, 56.2, 49.4, 74.5,